# Day 036 · 滚动窗口 rolling · 中国版
**Rolling Windows** · 阶段 P2 · Python 量化工具栈

> 今天讲滚动窗口 rolling,这是所有技术指标的共同地基。你天天在行情软件上看到的 20 日均线、60 日均线、布林带、MACD,本质上全是用 rolling 这一个工具算出来的。学会了它,你就不用再死记每个指标的公式,而是能亲手造出任何指标。这一节做五件事 — 第一,讲 rolling 滚动窗口到底是什么,怎么用它一句话算出 20 日均线;第二,讲均线的本质 —— 它其实就是「平滑加滞后」,搞懂这点你才明白为什么均线金叉总是慢半拍;第三,亲手造一条布林带,理解所谓上轨下轨就是均线上下加减两倍标准差;第四,讲 min_periods 怎么处理开头数据不足的问题,以及一个致命的概念 —— 前视偏差,也就是不小心偷看了未来数据;第五,讲 rolling 的两个兄弟 —— expanding 累计窗口和 ewm 指数加权,前者用来算历史最大回撤,后者是 EMA 和 MACD 的基础。我们用真实的茅台3 年日线,把均线、布林带、回撤亲手算一遍,所有指标都不靠现成库、自己用 rolling 造出来。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-05-29  ·  **建议学习时长:** 20 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [1]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


⏳ 缺少 1 个包,自动安装:['baostock']
✓ 安装完成
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [2]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 搞懂 rolling 滚动窗口 — 在时间序列上开一个固定宽度的窗口沿时间滑动,每个位置算窗口内的统计量,20 日均线就是 rolling(20).mean()
- 理解均线的本质是平滑加滞后 — 均线把价格噪音抹平(平滑),代价是反应慢半拍(滞后),窗口越长越平滑也越滞后,这解释了为什么金叉死叉总是慢一拍
- 亲手造布林带 — 中轨是 20 日均线,上下轨是中轨加减 2 倍滚动标准差,理解它衡量的是价格偏离均值的程度
- 掌握 min_periods 和前视偏差 — 知道怎么处理开头窗口不满的 NaN,更要理解 rolling 默认只用过去数据不会偷看未来,但自己 shift 错方向就会造成致命的未来函数
- 学会 expanding 和 ewm 两兄弟 — expanding 是累计窗口(算历史最大回撤靠它),ewm 是指数加权移动平均(EMA 比普通均线更跟手,是 MACD 的基础)

## 历史背景:老李迷信均线金叉,却总在金叉后被套

讲一个典型散户老李的困惑。老李是均线信徒,他的交易规则很简单:当 5 日均线上穿 20 日均线(所谓「金叉」)就买入,下穿(「死叉」)就卖出。这套规则听起来很科学,书上、视频里到处都在讲。但老李实操下来,发现自己总是在金叉买入后没多久就被套,死叉卖出后股价又涨回去,来回挨打。他怀疑是不是均线参数没调好,把 5 日、20 日换成 10 日、30 日,结果还是一样。

问题的根子,在于老李没理解均线的本质。均线不是什么神秘的预测工具,它就是「过去 N 天收盘价的平均」。既然是平均过去,它天然就有两个特性:一是平滑,把每天上蹿下跳的噪音抹平,曲线变得很顺滑;二是滞后,因为它永远在「回头看」过去 N 天,所以对最新的变化反应总是慢半拍。窗口取得越长(比如 60 日),越平滑,但也越滞后。

这就解释了老李的遭遇:当金叉真正发生时,价格其实已经涨了一段了(均线滞后),他追进去往往买在阶段高点附近;等死叉出现,价格也已经跌了一段,他割在低点附近。均线金叉死叉本质是「追涨杀跌」的滞后信号,在震荡市里会被反复打脸。这不是参数问题,是均线作为滞后指标的固有局限。

那有没有办法让均线跟手一点?有,这就是今天要讲的 EMA 指数加权移动平均。普通均线(SMA)对过去 N 天一视同仁,每天权重一样;而 EMA 给越近的日子越高的权重,所以它对最新变化反应更快、滞后更小。大名鼎鼎的 MACD 指标,核心就是两条 EMA 的差。理解了 SMA 和 EMA 的区别,老李就能明白自己工具的脾气,而不是盲目迷信。

今天我们就用真实的茅台数据,亲手用 rolling 造出均线、用滚动标准差造出布林带、用 expanding 算出历史最大回撤、用 ewm 造出 EMA,把这些天天听说却一知半解的指标,从最底层的原理彻底搞通。学完这一节,所有技术指标在你眼里都不再是黑箱,你能自己造、自己改、看懂它们的脾气和局限。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. rolling 滚动窗口 — 一个会移动的取数框

rolling 是今天的主角,理解它只需要一个画面:想象在你的价格曲线上,放一个固定宽度的「取数框」,比如宽 20 天。这个框从左往右一天一天地滑动,每滑到一个位置,就把框里这 20 天的数据拿出来算个统计量(平均、标准差、最大值都行),算完的结果记在框的右边缘那一天上。框滑过整条曲线,就得到一条新的曲线。这就是滚动窗口。

最经典的应用就是移动平均线。20 日均线,就是这个宽 20 天的框,每个位置算框内 20 天收盘价的平均值。代码极其简洁:close.rolling(20).mean(),一句话就出来了。把 mean 换成 std 就是滚动标准差,换成 max 就是滚动最高价(N 日新高就靠它判断),换成 min 就是滚动最低价。一个 rolling,配上不同的统计量,就能造出一大堆指标。

有个关键细节关乎「会不会作弊」:Pandas 的 rolling 默认是「右对齐」的,也就是每个位置的窗口,只包含「当天和过去 N 减 1 天」的数据,绝不包含未来。这一点至关重要 —— 它保证了你算出来的指标在历史上每一天都是「当时真实能看到的」,不会偷看未来。这是 rolling 能安全用于回测的根本前提。

一句话:rolling 是一个沿时间滑动的取数框,配 mean/std/max/min 就能造出绝大多数技术指标,而且默认只看过去、不偷看未来。


### 2. 均线的本质 — 平滑与滞后这对孪生兄弟

很多人把均线当成某种神秘的趋势预测,这是天大的误解。均线的本质朴素到不能再朴素:它就是过去 N 天收盘价的简单平均。理解了「平均过去」这四个字,均线的一切脾气都能推导出来。

第一个脾气是平滑。单看每天的收盘价,上蹿下跳充满噪音,趋势很难看清。一平均,那些随机的上下波动互相抵消,曲线立刻变得顺滑,大趋势浮现出来。这是均线的价值 —— 帮你过滤噪音、看清趋势。

第二个脾气是滞后,这是平滑的必然代价。因为均线永远在「回头看」过去 N 天,它对最新发生的变化反应必然慢半拍。价格刚开始涨,均线还被前面那些低价拖着,得过一阵才反应过来掉头向上。窗口越长,纳入的旧数据越多,越平滑,但也越滞后 —— 60 日均线比 20 日均线平滑得多,也迟钝得多。

平滑和滞后是一对孪生兄弟,你不可能只要平滑不要滞后,这是均线作为「回看型」指标的数学宿命。这就解释了开头老李的遭遇:均线金叉时价格往往已涨了一段,死叉时往往已跌了一段,所以金叉死叉本质是滞后的追涨杀跌信号,在震荡市里会反复打脸。明白这一点,你就不会再盲目迷信均线,而是把它当成一个「有滞后的趋势过滤器」来恰当使用。


### 3. 布林带 — 均线上下加减两倍标准差

布林带是出镜率极高的指标,听起来高级,拆开看其实就是 rolling 的两个简单组合,自己几行就能造出来。

它由3 条线组成。中轨,就是我们刚学的 20 日均线,代表近期的价格中枢。上轨,是中轨加上 2 倍的「20 日滚动标准差」。下轨,是中轨减去 2 倍的 20 日滚动标准差。滚动标准差也是 rolling 算的 —— close.rolling(20).std(),它衡量的是最近 20 天价格的波动有多大。波动大,标准差大,上下轨就张得开;波动小,上下轨就收得窄。所以布林带是会「呼吸」的:行情剧烈时变宽,行情平静时收窄。

为什么用 2 倍标准差?这背后是统计学的常识:如果价格波动近似正态分布,那么大约 95% 的数据会落在「均值 ± 2 倍标准差」的范围内。所以理论上,价格大约 95% 的时间会待在布林带里,只有约 5% 的时间会突破上下轨。这给了布林带一个朴素的解读:价格触碰上轨,说明它相对近期中枢偏高了(可能超买);触碰下轨,说明偏低了(可能超卖)。注意这只是统计上的「偏离程度」描述,不是必涨必跌的信号 —— 强趋势里价格可以贴着上轨一路涨,这是新手最容易误用的地方。

今天我们会亲手算一遍,并验证一下:真实的茅台数据里,价格落在布林带内的比例,是不是真的接近理论上的 95%。这种「用真实数据验证统计学预期」的习惯,是量化研究员的基本功。


### 4. min_periods 与前视偏差 — 开头的 NaN 和偷看未来的大忌

用 rolling 有两个绕不开的实务问题,一个是技术细节,一个是致命大忌。

先说技术细节:开头的 NaN。20 日均线需要 20 天数据才能算第一个值,所以前 19 天因为窗口还没填满,结果都是 NaN(空值)。这是正常的,但有时你不想浪费开头这段数据,可以用 min_periods 参数。比如 rolling(20, min_periods=5),意思是「只要窗口里攒够 5 个值就开始算」,这样前 4 天才是 NaN,第 5 天起就有(不太准但聊胜于无的)均线值了。min_periods 让你在「数据完整性」和「尽早出值」之间做权衡。

再说致命大忌:前视偏差,也叫「未来函数」。这是量化里最严重、最隐蔽的错误之一 —— 在计算历史上某一天的指标时,不小心用到了那一天之后才会发生的数据。好消息是,Pandas 的 rolling 默认右对齐、只用过去数据,本身不会造成前视偏差。坏消息是,一旦你开始手工对数据做位移(shift),就极容易出错。shift(1) 是把数据往后挪一天(用昨天的值,安全);而 shift(-1) 是把数据往前挪一天 —— 等于让今天用上了明天的数据,这就是赤裸裸地偷看未来。

前视偏差的可怕在于:它会让你的回测结果好得不真实,因为你的「策略」实际上开了天眼,而实盘里你永远拿不到未来数据,一上实盘就原形毕露、巨亏。所以记住这条铁律:做任何涉及时间位移的操作,反复确认你只用了「当下及过去」的数据,shift 的方向尤其要盯死。


### 5. expanding 和 ewm — rolling 的两个亲兄弟

学完 rolling,再认识它的两个亲兄弟,工具箱就齐了。

第一个是 expanding,累计窗口。rolling 的窗口宽度是固定的(永远 20 天),而 expanding 的窗口是「从第一天一直累计到当前」,越往后窗口越大。它最经典的用途是算「历史最高净值」—— expanding().max() 会给出「从开始到今天为止见过的最高点」。有了历史最高,再用「当前净值除以历史最高减一」,就得到回撤曲线,其中最深的那个点就是最大回撤,这是衡量一个策略「最惨时亏多少」的核心风险指标。今天我们就用它算茅台的历史最大回撤。

第二个是 ewm,指数加权移动平均。普通均线(SMA)对窗口内的每一天一视同仁,权重都一样。而 ewm 算出来的 EMA(指数移动平均)给越近的日子越高的权重,越远的日子权重指数级衰减。这么做的好处是 EMA 对最新变化反应更快、滞后更小,比 SMA 更「跟手」。它的衰减速度由一个 span(跨度)参数控制,span 越小越敏感。大名鼎鼎的 MACD 指标,核心就是「快线 EMA 减慢线 EMA」。理解了 EMA 比 SMA 跟手,你就明白了为什么很多短线指标偏爱 EMA。

一句话总结三兄弟:rolling 是固定宽度的滑动窗口(造均线、布林带),expanding 是从头累计的窗口(造历史峰值、最大回撤),ewm 是给近期加权的指数窗口(造 EMA、MACD)。它们仨,撑起了几乎所有技术指标。


## 实操:滚动窗口五件套 — rolling/均线/布林带/min_periods/expanding/ewm(中国版 — baostock,跟原版相同)

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [3]:
# day_036_rolling.py — 滚动窗口:rolling/均线/布林带/min_periods/expanding/ewm(中国版 — baostock,跟原版相同)
# 用真实 A 股日线,亲手造 20 日均线和布林带,讲透所有技术指标的共同地基
# 数据:baostock 日线(免费、稳定、国内零翻墙)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs

pd.set_option('display.width', 140)
CODE = 'sh.600519'
NAME = '茅台'
START = '2022-01-01'
END = '2024-12-31'
WIN = 20                            # 主窗口:20 日(约一个月交易日)

# ============ 0. 数据拉取:日线收盘价 ============
print('==== 0. 数据拉取(baostock 日线)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
rs = bs.query_history_k_data_plus(
    CODE, 'date,close', start_date=START, end_date=END,
    frequency='d', adjustflag='2')
rows = []
while (rs.error_code == '0') and rs.next():
    rows.append(rs.get_row_data())
bs.logout()
df = pd.DataFrame(rows, columns=['date', 'close'])
df['date'] = pd.to_datetime(df['date'])
df['close'] = df['close'].astype(float)
df = df.set_index('date').sort_index()
close = df['close']
print(f'{NAME} 日线 {len(close)} 条  {close.index[0].date()} → {close.index[-1].date()}')

# ============ 1. rolling:20 日均线 ============
print('\n==== 1. rolling 滚动窗口 → 20 日均线 ====')
ma20 = close.rolling(WIN).mean()
ma60 = close.rolling(60).mean()
# rolling 默认右对齐:每个点只用「当天 + 过去 19 天」,绝不偷看未来
print(f'20 日均线 前 {WIN-1} 个值是 NaN(窗口还没填满):前 3 个 = {ma20.head(3).round(2).tolist()}')
print(f'第 {WIN} 天起才有第一个均线值 = {ma20.iloc[WIN-1]:.2f}')
# 手工校验:第 20 天均线 == 前 20 天收盘的简单平均
manual = close.iloc[:WIN].mean()
print(f'手工校验:前 {WIN} 天收盘均值 {manual:.2f} == rolling 结果 {ma20.iloc[WIN-1]:.2f} ?', np.isclose(manual, ma20.iloc[WIN-1]))

# ============ 2. 均线的本质:平滑 + 滞后 ============
print('\n==== 2. 平滑 vs 滞后 ====')
# 用「波动率」量化平滑程度:均线的日变化标准差应远小于原价
vol_close = close.pct_change().std()
vol_ma20 = ma20.pct_change().std()
print(f'原价 日变化波动 {vol_close*100:.2f}%')
print(f'20 日均线 日变化波动 {vol_ma20*100:.2f}%(平滑了 {(1-vol_ma20/vol_close)*100:.0f}%,但代价是滞后)')

# ============ 3. 布林带:中轨 ± 2 倍滚动标准差 ============
print('\n==== 3. 布林带 ====')
mid = close.rolling(WIN).mean()                 # 中轨 = 20 日均线
std20 = close.rolling(WIN).std()                 # 20 日滚动标准差(样本 std)
upper = mid + 2 * std20                           # 上轨
lower = mid - 2 * std20                           # 下轨
# 统计学预期:价格约 95% 时间落在上下轨之间
inside = ((close <= upper) & (close >= lower))
inside_pct = inside[mid.notna()].mean()
print(f'价格落在布林带内的比例 = {inside_pct*100:.1f}%(理论 ±2σ ≈ 95%)')
touch_up = (close > upper).sum()
touch_lo = (close < lower).sum()
print(f'突破上轨 {touch_up} 次,跌破下轨 {touch_lo} 次')

# ============ 4. min_periods:开头 NaN 怎么办 + 前视偏差 ============
print('\n==== 4. min_periods ====')
ma20_mp = close.rolling(WIN, min_periods=5).mean()   # 至少 5 个值就开始算
print(f'默认 rolling 前 {WIN-1} 个是 NaN;min_periods=5 后,前 4 个才 NaN:{ma20_mp.head(6).round(1).tolist()}')
print('关键:rolling 默认右对齐,只用过去数据 → 不会未来函数;但自己 shift 要小心方向,shift(-1) 会偷看未来')

# ============ 5. expanding 累计窗口 → 算历史回撤 ============
print('\n==== 5. expanding 累计窗口 ====')
nav = (1 + close.pct_change().fillna(0)).cumprod()   # 净值
run_max = nav.expanding().max()                       # 累计历史最高(从头到当前)
drawdown = nav / run_max - 1                            # 回撤
mdd = drawdown.min()
mdd_date = drawdown.idxmin()
print(f'expanding().max() 算出历史峰值,最大回撤 = {mdd*100:.1f}%(发生在 {mdd_date.date()})')

# ============ 6. ewm 指数加权:EMA 比 SMA 更跟手 ============
print('\n==== 6. ewm 指数加权移动平均 ====')
ema20 = close.ewm(span=WIN, adjust=False).mean()      # 20 日 EMA,近期权重大
# 比较 SMA 和 EMA 对最近一次价格变化的反应快慢(平均滞后)
lag_sma = (close - ma20).abs().mean()
lag_ema = (close - ema20).abs().mean()
print(f'价格与 20 日 SMA 平均距离 {lag_sma:.2f}')
print(f'价格与 20 日 EMA 平均距离 {lag_ema:.2f}(EMA 给近期加权,更贴价格、滞后更小)')

# ============ 7. 出图 ============
# chart_01:收盘价 + 20/60 日均线
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(close.index, close, color='#999999', lw=0.8, label='收盘价')
ax.plot(ma20.index, ma20, color='#1f77b4', lw=1.5, label='20 日均线')
ax.plot(ma60.index, ma60, color='#d62728', lw=1.5, label='60 日均线')
ax.set_title(f'{NAME} 收盘价 + 双均线(rolling 滚动窗口)')
ax.set_ylabel('价格(元)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('chart_ma.png', dpi=130); plt.close(fig)

# chart_02:布林带
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(close.index, close, color='#333333', lw=1, label='收盘价')
ax.plot(mid.index, mid, color='#1f77b4', lw=1.2, label='中轨(20 日均线)')
ax.plot(upper.index, upper, color='#2ca02c', lw=1, ls='--', label='上轨 (+2σ)')
ax.plot(lower.index, lower, color='#d62728', lw=1, ls='--', label='下轨 (-2σ)')
ax.fill_between(mid.index, lower, upper, color='#1f77b4', alpha=0.08)
ax.set_title(f'{NAME} 布林带(中轨 ± 2 倍滚动标准差,约 {inside_pct*100:.0f}% 价格落带内)')
ax.set_ylabel('价格(元)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('chart_bollinger.png', dpi=130); plt.close(fig)

# chart_03:SMA vs EMA 对比(取最近一段看得清)
recent = close.index >= (close.index[-1] - pd.Timedelta(days=200))
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(close.index[recent], close[recent], color='#999999', lw=0.9, label='收盘价')
ax.plot(ma20.index[recent], ma20[recent], color='#1f77b4', lw=1.6, label='20 日 SMA(简单)')
ax.plot(ema20.index[recent], ema20[recent], color='#ff7f0e', lw=1.6, label='20 日 EMA(指数加权)')
ax.set_title(f'{NAME} 最近 200 天:SMA vs EMA(EMA 更跟手)')
ax.set_ylabel('价格(元)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('chart_sma_ema.png', dpi=130); plt.close(fig)

# chart_04:expanding 净值 + 历史峰值 + 回撤
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(nav.index, nav, color='#1f77b4', lw=1.2, label='净值')
ax1.plot(run_max.index, run_max, color='#d62728', lw=1, ls='--', label='历史峰值 expanding().max()')
ax1.set_ylabel('净值'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title(f'{NAME} 净值 vs 历史峰值(expanding 累计窗口)')
ax2.fill_between(drawdown.index, drawdown*100, 0, color='#d62728', alpha=0.4)
ax2.set_ylabel('回撤 (%)'); ax2.grid(alpha=0.3)
ax2.set_title(f'回撤曲线(最大回撤 {mdd*100:.1f}%)')
fig.tight_layout(); fig.savefig('chart_drawdown.png', dpi=130); plt.close(fig)

print('\n[done] 4 张图已保存,output.txt 见上方全部打印')

==== 0. 数据拉取(baostock 日线)====
login success!
logout success!
茅台 日线 726 条  2022-01-04 → 2024-12-31

==== 1. rolling 滚动窗口 → 20 日均线 ====
20 日均线 前 19 个值是 NaN(窗口还没填满):前 3 个 = [nan, nan, nan]
第 20 天起才有第一个均线值 = 1716.74
手工校验:前 20 天收盘均值 1716.74 == rolling 结果 1716.74 ? True

==== 2. 平滑 vs 滞后 ====
原价 日变化波动 1.67%
20 日均线 日变化波动 0.34%(平滑了 80%,但代价是滞后)

==== 3. 布林带 ====
价格落在布林带内的比例 = 88.8%(理论 ±2σ ≈ 95%)
突破上轨 34 次,跌破下轨 45 次

==== 4. min_periods ====
默认 rolling 前 19 个是 NaN;min_periods=5 后,前 4 个才 NaN:[nan, nan, nan, nan, 1763.5, 1755.7]
关键:rolling 默认右对齐,只用过去数据 → 不会未来函数;但自己 shift 要小心方向,shift(-1) 会偷看未来

==== 5. expanding 累计窗口 ====
expanding().max() 算出历史峰值,最大回撤 = -34.5%(发生在 2024-09-19)

==== 6. ewm 指数加权移动平均 ====
价格与 20 日 SMA 平均距离 42.23
价格与 20 日 EMA 平均距离 35.60(EMA 给近期加权,更贴价格、滞后更小)

[done] 4 张图已保存,output.txt 见上方全部打印


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股散户 | 任意个股的均线金叉死叉 | 散户最熟悉的 5 日上穿 20 日「金叉」买入、下穿「死叉」卖出,本质就是两条 rolling 均线的交叉。但因为均线滞后,金叉时价格往往已涨一段、死叉时已跌一段,震荡市里反复被套。理解均线平滑加滞后的本质,才能不盲目迷信金叉死叉,这正是这一节故事的原型。 |
| 技术分析 / 波段交易 | 活跃个股 / 指数的布林带 | 布林带超买超卖是波段交易者的常用工具:价格触上轨视为偏高、触下轨视为偏低。但强趋势里价格能贴着上轨一路涨,机械地「触上轨就卖」会过早下车。布林带衡量的是偏离程度而非必涨必跌信号,这是新手最容易误用的地方,用 rolling 亲手造一遍就明白它的脾气。 |
| 量化研究 | 全市场技术因子库 | 量化研究里几乎所有技术类因子 —— 动量、波动率、N 日新高新低、均线乖离率等等 —— 底层都是 rolling 的各种变体(rolling.mean/std/max/min/apply)。掌握 rolling 一个工具,就能批量造出几十个技术因子,这是因子工程里最基础也最高频的1 块。 |
| 风险管理 / 回测 | 任意策略的净值曲线 | 评价一个策略「最惨时亏多少」,核心指标是最大回撤,算法就是 expanding().max() 求历史峰值,再算当前相对峰值的跌幅。任何回测报告、任何基金净值分析都绕不开它。今天我们用 expanding 给茅台净值算最大回撤,就是这套风控分析的最小实现。 |
| 短线 / MACD 体系 | 指数或个股的 MACD / EMA | MACD 是流传最广的短线指标之一,它的核心是两条不同周期的 EMA 之差,而 EMA 正是 ewm 算出来的。EMA 给近期加权、比 SMA 跟手,所以 MACD 比纯 SMA 交叉反应更快。理解了 ewm,你就理解了 MACD、DEA、DIF 这一整套指标的数学来源,不再死记。 |


## 常见坑

### ⚠ 01. rolling 忘了处理开头 NaN — 数据被误删或报错

20 日均线开头 19 个值是 NaN,不少人直接 dropna 把这段连同原始数据一起删了,或者后续运算碰到 NaN 报错/被传染。**正确做法**:清楚开头 NaN 是正常的;不想浪费开头数据用 min_periods 让它早点出值;真要删只删指标列的 NaN,别把原始价格也误删;运算前确认 NaN 已妥善处理。

### ⚠ 02. 把均线当预测工具 — 迷信金叉死叉

均线是「过去 N 天的平均」,是滞后指标不是领先指标。把它当成能预测未来的神器,迷信金叉买死叉卖,在震荡市里会被反复打脸(金叉时已涨一段、死叉时已跌一段)。**正确做法**:把均线理解为「有滞后的趋势过滤器」,清楚它在趋势市好用、震荡市失灵,配合其他条件用,绝不单凭金叉死叉无脑操作。

### ⚠ 03. 前视偏差 / 未来函数 — shift 方向错偷看未来

rolling 默认只用过去数据是安全的,但一旦手工 shift 就容易出错:shift(-1) 把未来数据挪到了今天,等于开天眼。回测结果会好得离谱,一上实盘原形毕露巨亏。**正确做法**:任何时间位移操作反复确认只用了「当下及过去」的数据;shift(1) 用昨天值是安全的、shift(-1) 是偷看未来;回测好得不真实时,第一个就排查前视偏差。

### ⚠ 04. 布林带标准差用错或窗口太短 — 带子忽宽忽窄不可信

滚动标准差窗口取得太短(比如 5 天),标准差估计极不稳定,布林带忽宽忽窄;另外样本标准差(ddof=1)和总体标准差(ddof=0)会有细微差别,不同软件口径不一致。**正确做法**:布林带标准差窗口跟中轨一致(通常 20 天),别太短;清楚自己用的是样本还是总体标准差,跨工具对比时统一口径。

### ⚠ 05. 混淆 ewm 的 span / alpha / com 参数 — EMA 算出来不对

ewm 控制衰减速度有 span(跨度)、alpha(平滑系数)、com(质心)好几个参数,只能给一个,给混了或理解错,算出来的 EMA 周期跟你想的不一样。还有人把 EMA 当成和 SMA 一回事去理解。**正确做法**:常用 span(跟 SMA 的窗口天数可类比,如 span=20 近似 20 日);adjust=False 取递推式 EMA(跟交易软件口径一致);清楚 EMA 给近期加权、跟 SMA 不同。

## 实战 SOP · 滚动窗口实战 7 条 SOP

1. 均线/滚动指标用 close.rolling(N).mean()/std()/max() — 一个 rolling 配不同统计量造出绝大多数技术指标
2. 记住均线是平滑加滞后 — 窗口越长越平滑也越滞后,均线是趋势过滤器不是预测器
3. 布林带 = 20 日均线 ± 2 倍 20 日滚动标准差 — 衡量偏离程度,约 95% 价格落带内,触轨不等于必反转
4. 开头 NaN 是正常的 — 不想浪费数据用 min_periods,删 NaN 只删指标列别误删原始价
5. 盯死前视偏差 — rolling 默认安全只看过去;手工 shift 时 shift(-1) 偷看未来,回测好得离谱先查这
6. 最大回撤用 expanding().max() 求历史峰值 — 当前净值除以历史峰值减一,最深处就是最大回撤
7. 要更跟手的均线用 ewm(span=N, adjust=False) 算 EMA — EMA 给近期加权、滞后更小,是 MACD 的基础

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. rolling 是沿时间滑动的固定宽度取数框 — 配 mean/std/max/min 造出绝大多数技术指标,默认右对齐只看过去不偷看未来
3. 均线的本质是平滑加滞后 — 平均过去 N 天既抹平噪音又反应慢半拍,窗口越长越平滑也越滞后,金叉死叉是滞后信号
4. 布林带 = 中轨(20 日均线)± 2 倍滚动标准差 — 会随波动呼吸,约 95% 价格落带内,衡量偏离程度而非必涨必跌
5. min_periods 控制开头不满窗口的 NaN — 想早点出值就设,开头 NaN 本身是正常现象
6. 前视偏差是最隐蔽的致命错 — rolling 默认安全,但 shift(-1) 会偷看未来,导致回测虚高、实盘巨亏
7. expanding 是从头累计的窗口 — expanding().max() 求历史峰值,进而算回撤,最大回撤是核心风险指标
8. ewm 算 EMA 给近期加权 — 比 SMA 更跟手、滞后更小,是 MACD 等短线指标的数学基础

## 自测题

**Q1.** rolling 滚动窗口是什么?20 日均线该怎么用一句话写出来?为什么说它默认不会偷看未来?(提示:沿时间滑动的固定宽度取数框;close.rolling(20).mean();默认右对齐只含当天和过去)

**Q2.** 为什么均线总是慢半拍?平滑和滞后是什么关系?窗口长短对它们有什么影响?(提示:均线是过去 N 天平均、永远回头看;平滑必然带来滞后;窗口越长越平滑也越滞后)

**Q3.** 布林带的3 条线分别怎么算?为什么用 2 倍标准差?价格触碰上轨是不是就该卖?(提示:中轨 20 日均线、上下轨 ±2 倍滚动标准差;正态下约 95% 落 ±2σ 内;触轨是偏离描述非必反转,强趋势会贴轨走)

**Q4.** 什么是前视偏差(未来函数)?为什么 rolling 默认安全而 shift(-1) 危险?它会导致什么后果?(提示:用到了当天之后的数据;rolling 右对齐只看过去、shift(-1) 把未来挪到今天;回测虚高、实盘巨亏)

**Q5.** expanding 和 rolling 有什么区别?怎么用 expanding 算最大回撤?EMA 比 SMA 强在哪?(提示:expanding 窗口从头累计、rolling 固定宽度;expanding().max() 求历史峰值再算跌幅;EMA 给近期加权更跟手、滞后小)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 037 · 多重索引与 groupby** (MultiIndex & GroupBy)

第 37 课 多重索引与 groupby。今天我们在单只股票的时间序列上开窗口,但真实研究常常要同时处理几百只股票、还要按行业分组算因子。这就需要 MultiIndex 多重索引(给数据加多层标签)和 groupby 分组聚合(按行业、按日期把数据分堆处理)。明天讲怎么用这两个工具处理「面板数据」,以及量化里极其常用的「行业中性化」操作,这是从单票分析迈向全市场研究的关键一步。

## 推荐阅读

- Wes McKinney《Python for Data Analysis》(2022 第 3 版,O'Reilly)— 第 11 章末尾专讲 rolling / expanding / ewm 窗口函数,Pandas 作者亲笔,本节最对口的权威章节
- Pandas 官方文档《Windowing operations》(免费在线,pandas.pydata.org)— rolling/expanding/ewm 的权威参考,各种参数(min_periods/center/win_type)和 ewm 的 span/alpha/com 讲得最准
- John Bollinger《Bollinger on Bollinger Bands》(2001,McGraw-Hill)— 布林带发明人本人写的专著,讲透布林带的设计初衷和正确用法,纠正「触轨就反转」的常见误用
- Marcos López de Prado《Advances in Financial Machine Learning》(2018,Wiley)— 第 1、3 章反复强调前视偏差(未来函数)对回测的毁灭性影响,是理解为什么必须只用过去数据的权威来源
- Python 工具栈 — Pandas(rolling 核心)、TA-Lib(C 写的高速技术指标库)、pandas-ta(纯 Python 技术指标)、numba(给 rolling.apply 自定义函数 JIT 加速十几倍),这四个是技术指标计算的进阶生态